# Figuring out what's wrong with HRDPS 1km results

In [1]:
import xarray as xr
import numpy as np
import pandas as pd


# ============================================================
# Weight files
# ============================================================

path_weights_casr = (
    '/home/jqiu/analysis-junqi/Analysis_Atmospheric_Forcing/'
    'Data_weights/weights-HRDPS-1km_202108.nc'
)

path_weights_hrdps = (
    '/home/jqiu/analysis-junqi/Analysis_Atmospheric_Forcing/'
    'Data_weights/weights-continental2.5-hrdps.nc'
)

ds_weights_casr = xr.open_dataset(path_weights_casr)
ds_weights_hrdps = xr.open_dataset(path_weights_hrdps)


# ============================================================
# Stations
# ============================================================

stations = {
    'Pam Rocks': {
        'y': 502,
        'x': 341,
        'lat': 49.487434,
        'lon': -123.30153,
    },

    'Sand Heads': {
        'y': 428,
        'x': 293,
        'lat': 49.107586,
        'lon': -123.30203,
    },
}


# ============================================================
# Function
# ============================================================

def get_source_and_neighbors(weights, y, x, station_name=None):
    """
    For one NEMO target grid point:

    1. Find the 4 atmospheric source points actually used
       by src01..src04 / wgt01..wgt04.

    2. For each of those 4 source points, find its 8 neighboring
       atmospheric grid points.

    3. Remove duplicates.

    Returns
    -------
    df : pandas.DataFrame

        Columns include:
            point_type
            source_number
            atmos_ilat
            atmos_ilon
            atmos_lat
            atmos_lon
            src
            weight
    """

    nlat = weights.sizes["lat"]
    nlon = weights.sizes["lon"]

    # --------------------------------------------------------
    # Step 1: get the 4 source points actually used by NEMO
    # --------------------------------------------------------

    used_points = {}

    for k in range(1, 5):

        src = int(
            weights[f"src0{k}"]
            .isel(y=y, x=x)
            .item()
        )

        weight = float(
            weights[f"wgt0{k}"]
            .isel(y=y, x=x)
            .item()
        )

        # Fortran 1-based -> Python 0-based
        idx = src - 1

        ilat = idx // nlon
        ilon = idx % nlon

        lat = float(
            weights["nav_lat"]
            .isel(lat=ilat, lon=ilon)
            .item()
        )

        lon = float(
            weights["nav_lon"]
            .isel(lat=ilat, lon=ilon)
            .item()
        )

        used_points[(ilat, ilon)] = {
            "station": station_name,
            "nemo_y": y,
            "nemo_x": x,

            "point_type": "used",
            "source_number": k,

            "src": src,
            "atmos_ilat": ilat,
            "atmos_ilon": ilon,

            "atmos_lat": lat,
            "atmos_lon": lon,

            "weight": weight,
        }


    # --------------------------------------------------------
    # Step 2: find 8 neighbors around EACH used source point
    # --------------------------------------------------------

    neighbor_points = {}

    # 8-connected neighborhood
    offsets = [
        (-1, -1),
        (-1,  0),
        (-1, +1),

        ( 0, -1),
        ( 0, +1),

        (+1, -1),
        (+1,  0),
        (+1, +1),
    ]

    for (ilat0, ilon0), source_info in used_points.items():

        source_number = source_info["source_number"]

        for dy, dx in offsets:

            ilat = ilat0 + dy
            ilon = ilon0 + dx

            # -----------------------------------------------
            # Skip points outside atmospheric grid
            # -----------------------------------------------

            if not (0 <= ilat < nlat):
                continue

            if not (0 <= ilon < nlon):
                continue


            # -----------------------------------------------
            # If this neighbor itself is one of the 4 used
            # points, don't add it as a neighbor
            # -----------------------------------------------

            if (ilat, ilon) in used_points:
                continue


            # -----------------------------------------------
            # Atmospheric coordinates
            # -----------------------------------------------

            lat = float(
                weights["nav_lat"]
                .isel(lat=ilat, lon=ilon)
                .item()
            )

            lon = float(
                weights["nav_lon"]
                .isel(lat=ilat, lon=ilon)
                .item()
            )


            # Python grid index -> Fortran linear index
            src = ilat * nlon + ilon + 1


            # -----------------------------------------------
            # Avoid duplicates
            #
            # The same neighbor may be adjacent to two or
            # more of the 4 interpolation points.
            # -----------------------------------------------

            key = (ilat, ilon)

            if key not in neighbor_points:

                neighbor_points[key] = {
                    "station": station_name,
                    "nemo_y": y,
                    "nemo_x": x,

                    "point_type": "neighbor",

                    # not one of src01..04
                    "source_number": np.nan,

                    "src": src,

                    "atmos_ilat": ilat,
                    "atmos_ilon": ilon,

                    "atmos_lat": lat,
                    "atmos_lon": lon,

                    # neighbor does NOT participate
                    # in the interpolation
                    "weight": np.nan,

                    # useful to know which used point
                    # introduced this neighbor
                    "neighbor_of": [source_number],
                }

            else:

                # This neighbor touches more than one
                # of the four source points
                neighbor_points[key]["neighbor_of"].append(
                    source_number
                )


    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    records = (
        list(used_points.values())
        + list(neighbor_points.values())
    )

    df = pd.DataFrame(records)

    # Sort: used points first, then atmospheric grid location
    df["_sort_type"] = df["point_type"].map({
        "used": 0,
        "neighbor": 1,
    })

    df = (
        df
        .sort_values(
            ["_sort_type", "atmos_ilat", "atmos_ilon"]
        )
        .drop(columns="_sort_type")
        .reset_index(drop=True)
    )

    return df


# ============================================================
# Pretty print
# ============================================================

def print_source_and_neighbors(weights, y, x, station_name=None):

    df = get_source_and_neighbors(
        weights,
        y=y,
        x=x,
        station_name=station_name,
    )

    used = df[df["point_type"] == "used"]
    neighbors = df[df["point_type"] == "neighbor"]

    print()
    print("=" * 100)
    print(f"{station_name}")
    print(f"NEMO target: y={y}, x={x}")
    print("=" * 100)

    print()
    print("********** 4 atmospheric points actually USED **********")
    print()

    for _, row in used.iterrows():

        print(
            f"source {int(row['source_number'])}: "
            f"(ilat, ilon)=({int(row['atmos_ilat']):4d}, "
            f"{int(row['atmos_ilon']):4d})   "
            f"lat={row['atmos_lat']:10.6f}   "
            f"lon={row['atmos_lon']:11.6f}   "
            f"weight={row['weight']:.10f}"
        )

    print()
    print(
        "********** Neighboring atmospheric grid points "
        "(deduplicated) **********"
    )
    print()

    for _, row in neighbors.iterrows():

        print(
            f"(ilat, ilon)=({int(row['atmos_ilat']):4d}, "
            f"{int(row['atmos_ilon']):4d})   "
            f"lat={row['atmos_lat']:10.6f}   "
            f"lon={row['atmos_lon']:11.6f}   "
            f"neighbor_of={row['neighbor_of']}"
        )

    print()
    print("-" * 100)
    print(f"Number of USED points     = {len(used)}")
    print(f"Number of unique neighbors = {len(neighbors)}")
    print(f"Total plotted points       = {len(df)}")

    print(
        f"Sum of interpolation weights = "
        f"{used['weight'].sum():.12f}"
    )

    print()

    return df

In [2]:
# ============================================================
# CaSR
# ============================================================

print("\n######################## HRDPS 1km ########################")

casr_results = {}

for station_name, station in stations.items():

    casr_results[station_name] = print_source_and_neighbors(
        ds_weights_casr,
        y=station["y"],
        x=station["x"],
        station_name=station_name,
    )


# ============================================================
# HRDPS
# ============================================================

print("\n######################## HRDPS 2.5km ########################")

hrdps_results = {}

for station_name, station in stations.items():

    hrdps_results[station_name] = print_source_and_neighbors(
        ds_weights_hrdps,
        y=station["y"],
        x=station["x"],
        station_name=station_name,
    )


######################## HRDPS 1km ########################

Pam Rocks
NEMO target: y=502, x=341

********** 4 atmospheric points actually USED **********

source 4: (ilat, ilon)=(  -1, 1329)   lat= 60.295024   lon= 245.550000   weight=0.0000000000

********** Neighboring atmospheric grid points (deduplicated) **********

(ilat, ilon)=(   0, 1328)   lat= 50.064856   lon= 250.453856   neighbor_of=[4]
(ilat, ilon)=(   0, 1329)   lat= 50.066924   lon= 250.467424   neighbor_of=[4]

----------------------------------------------------------------------------------------------------
Number of USED points     = 1
Number of unique neighbors = 2
Total plotted points       = 3
Sum of interpolation weights = 0.000000000000


Sand Heads
NEMO target: y=428, x=293

********** 4 atmospheric points actually USED **********

source 4: (ilat, ilon)=(  -1, 1329)   lat= 60.295024   lon= 245.550000   weight=0.0000000000

********** Neighboring atmospheric grid points (deduplicated) **********

(ilat, ilon

**Something is seriously wrong with HRDPS 1km forcing field.** Maybe it's because they do not match at all.